## 3. Relacionar as tabelas

Esta etapa continua a partir de `dados_limpos`, sem alterar os DataFrames preparados acima.
O briefing 5 usa `orders`, `items`, `products`, `sellers` e `reviews`. A tabela de clientes não é necessária para estes relacionamentos.

O ponto de partida é a tabela de itens. Sua chave é formada por `order_id` e `order_item_id`.
Os cadastros de pedidos, produtos e vendedores devem ter uma linha por identificador.
As avaliações são agrupadas por `order_id` antes da junção, pois um pedido pode ter mais de uma avaliação.

In [41]:
# [IA] registro 1 do complemento das partes 3 e 4.
import pandas as pd

def conferir_chave(df, colunas, nome):
    if df[colunas].isna().any().any():
        raise ValueError(f'{nome}: há valores ausentes na chave {colunas}.')
    if df.duplicated(colunas).any():
        raise ValueError(f'{nome}: há chaves repetidas com dados diferentes em {colunas}.')

def relacionar_tabelas(dados_limpos):
    chaves = {
        'orders': ['order_id'],
        'items': ['order_id', 'order_item_id'],
        'products': ['product_id'],
        'sellers': ['seller_id'],
    }
    obrigatorias = set(chaves) | {'reviews'}
    faltantes = obrigatorias - set(dados_limpos)
    if faltantes:
        raise ValueError(f'Bases não carregadas: {sorted(faltantes)}')

    bases = {}
    duplicatas = []
    for nome in sorted(obrigatorias):
        df = dados_limpos[nome].copy()
        repetidas = int(df.duplicated().sum())
        bases[nome] = df.drop_duplicates().copy()
        duplicatas.append({'base': nome, 'linhas_entrada': len(df),
                           'duplicatas_completas': repetidas,
                           'linhas_utilizadas': len(bases[nome])})

    for nome, colunas in chaves.items():
        conferir_chave(bases[nome], colunas, nome)
    for coluna in ['order_id', 'seller_id', 'product_id']:
        if bases['items'][coluna].isna().any():
            raise ValueError(f'items: há valores ausentes em {coluna}.')

    orders, items = bases['orders'], bases['items']
    products, sellers = bases['products'], bases['sellers']
    reviews = bases['reviews']
    for coluna in ['order_purchase_timestamp', 'order_approved_at',
                   'order_delivered_carrier_date', 'order_delivered_customer_date',
                   'order_estimated_delivery_date']:
        orders[coluna] = pd.to_datetime(orders[coluna], format='mixed', errors='coerce')
    items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], format='mixed', errors='coerce')

    # review_id não é uma chave única global. A agregação é feita por pedido.
    reviews['review_score'] = pd.to_numeric(reviews['review_score'], errors='coerce')
    reviews['nota_valida'] = reviews['review_score'].where(reviews['review_score'].isin([1, 2, 3, 4, 5]))
    reviews['nota_baixa'] = reviews['nota_valida'].le(2) & reviews['nota_valida'].notna()
    avaliacoes = reviews.dropna(subset=['order_id']).groupby('order_id', as_index=False).agg(
        qtd_avaliacoes_pedido=('review_id', 'size'),
        qtd_avaliacoes_validas=('nota_valida', 'count'),
        nota_media_pedido=('nota_valida', 'mean'),
        menor_nota_pedido=('nota_valida', 'min'),
        maior_nota_pedido=('nota_valida', 'max'),
        qtd_avaliacoes_baixas_pedido=('nota_baixa', 'sum'),
    )
    conferir_chave(avaliacoes, ['order_id'], 'avaliações agregadas')

    # Contagens calculadas antes dos merges, para não depender de linhas multiplicadas.
    vendedores_por_pedido = items.groupby('order_id')['seller_id'].nunique()
    base_itens = items.copy()
    base_itens['qtd_vendedores_pedido'] = base_itens['order_id'].map(vendedores_por_pedido)
    historico_merges = []
    for nome, direita, chave, indicador in [
        ('items + orders', orders, 'order_id', 'pedido_encontrado'),
        ('items + products', products, 'product_id', 'produto_encontrado'),
        ('items + sellers', sellers, 'seller_id', 'vendedor_encontrado'),
        ('items + avaliações agregadas', avaliacoes, 'order_id', 'avaliacao_encontrada'),
    ]:
        antes = len(base_itens)
        base_itens = base_itens.merge(direita, on=chave, how='left', validate='many_to_one', indicator=True)
        sem_correspondencia = int(base_itens['_merge'].eq('left_only').sum())
        base_itens[indicador] = base_itens['_merge'].eq('both')
        base_itens = base_itens.drop(columns='_merge')
        historico_merges.append({'relacionamento': nome, 'chave': chave,
                                 'linhas_antes': antes, 'linhas_depois': len(base_itens),
                                 'itens_sem_correspondencia': sem_correspondencia})
        assert len(base_itens) == antes, f'O merge {nome} alterou a quantidade de itens.'

    conferir_chave(base_itens, ['order_id', 'order_item_id'], 'base de itens relacionada')
    auditoria = {
        'duplicatas': pd.DataFrame(duplicatas),
        'merges': pd.DataFrame(historico_merges),
        'pedidos_sem_itens': int((~orders['order_id'].isin(items['order_id'])).sum()),
        'pedidos_com_multiplas_avaliacoes': int(avaliacoes['qtd_avaliacoes_pedido'].gt(1).sum()),
        'reviews_sem_order_id': int(reviews['order_id'].isna().sum()),
        'reviews_sem_pedido_cadastrado': int((reviews['order_id'].notna() & ~reviews['order_id'].isin(orders['order_id'])).sum()),
        'notas_invalidas_ou_ausentes': int(reviews['nota_valida'].isna().sum()),
    }
    return base_itens, auditoria

In [42]:
# [IA] registro 1 do complemento das partes 3 e 4.
base_itens_relacionada, auditoria_relacionamentos = relacionar_tabelas(dados_limpos)

display(auditoria_relacionamentos['duplicatas'])
display(auditoria_relacionamentos['merges'])

print('Pedidos sem itens:', auditoria_relacionamentos['pedidos_sem_itens'])
print('Pedidos com mais de uma avaliação:', auditoria_relacionamentos['pedidos_com_multiplas_avaliacoes'])
print('Avaliações sem pedido cadastrado:', auditoria_relacionamentos['reviews_sem_pedido_cadastrado'])
print('Notas inválidas ou ausentes:', auditoria_relacionamentos['notas_invalidas_ou_ausentes'])

,base,linhas_entrada,duplicatas_completas,linhas_utilizadas
0,items,112650,0,112650
1,orders,99441,0,99441
2,products,32951,0,32951
3,reviews,99224,0,99224
4,sellers,3095,0,3095


,relacionamento,chave,linhas_antes,linhas_depois,itens_sem_correspondencia
0,items + orders,order_id,112650,112650,0
1,items + products,product_id,112650,112650,0
2,items + sellers,seller_id,112650,112650,0
3,items + avaliações agregadas,order_id,112650,112650,942


Pedidos sem itens: 775
Pedidos com mais de uma avaliação: 547
Avaliações sem pedido cadastrado: 0
Notas inválidas ou ausentes: 0


As junções usam `left` para preservar os itens e `validate='many_to_one'` para impedir a multiplicação de linhas por cadastros repetidos.
O histórico mostra quantas linhas existiam antes e depois de cada merge e quantos itens não encontraram correspondência.

`review_id` pode aparecer em pedidos diferentes e não foi usado para apagar avaliações.
Para cada pedido, a base mantém a quantidade de avaliações, a média, os extremos das notas e a quantidade de notas 1 ou 2.
Uma avaliação ausente permanece ausente; ela não recebe nota zero.

In [43]:
# [IA] registro 1 do complemento das partes 3 e 4.
colunas_conferencia = [
    'order_id', 'order_item_id', 'seller_id', 'product_id',
    'product_category_name', 'seller_state', 'price', 'freight_value',
    'qtd_avaliacoes_pedido', 'nota_media_pedido',
]
display(base_itens_relacionada[colunas_conferencia].head(10))

print('Itens relacionados:', len(base_itens_relacionada))
print('Pedidos distintos:', base_itens_relacionada['order_id'].nunique())
print('Vendedores distintos:', base_itens_relacionada['seller_id'].nunique())

,order_id,order_item_id,seller_id,product_id,product_category_name,seller_state,price,freight_value,qtd_avaliacoes_pedido,nota_media_pedido
0,00010242fe8c5a6d1ba2dd792cb16214,1,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,SP,58.90,13.29,1.0,5.0
1,00018f77f2f0320c557190d7a144bdd3,1,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f,pet_shop,SP,239.90,19.93,1.0,4.0
2,000229ec398224ef6ca0657da4fc703e,1,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,MG,199.00,17.87,1.0,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc,perfumaria,SP,12.99,12.79,1.0,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,PR,199.90,18.14,1.0,5.0
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,6426d21aca402a131fc0a5d0960a3c90,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,SP,21.90,12.69,1.0,4.0
6,00054e8431b9d7675808bcb819fb4a32,1,7040e82f899a04d1b434b795a43b4617,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia,SP,19.90,11.85,1.0,4.0
7,000576fe39319847cbb9d288c5617fa6,1,5996cddab893a4652a15592fb58ab8db,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,SP,810.00,70.75,1.0,5.0
8,0005a1a1728c9d785b8e2b08b904576c,1,a416b6a846a11724393025641d4edd5e,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,SP,145.95,11.65,1.0,1.0
9,0005f50442cb953dcd1d21e1fb923495,1,ba143b05f0110f0dc71ad71b4466ce92,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos,SP,53.99,11.40,1.0,4.0


Itens relacionados: 112650
Pedidos distintos: 98666
Vendedores distintos: 3095


## 4. Construir a tabela analítica

A tabela analítica tem **uma linha por `seller_id` e `order_id`**.
Assim, um vendedor com vários itens no mesmo pedido tem o valor dos itens somado, mas a nota do pedido aparece apenas uma vez para ele.
As categorias presentes no par vendedor–pedido ficam em `categorias`, separadas por `|`.

Pedidos com mais de um vendedor são identificados por `pedido_multivendedor`.
A data de despacho e a avaliação pertencem ao pedido na Olist, não a cada vendedor individualmente.
Esse limite precisa acompanhar a análise posterior.

In [44]:
# [IA] registro 1 do complemento das partes 3 e 4.
def construir_tabela_analitica(base_itens):
    itens = base_itens.copy()
    compra = itens['order_purchase_timestamp']
    despacho = itens['order_delivered_carrier_date']
    prazo = itens['shipping_limit_date']
    itens['despacho_antes_compra'] = despacho.notna() & compra.notna() & despacho.lt(compra)
    itens['prazo_antes_compra'] = prazo.notna() & compra.notna() & prazo.lt(compra)
    itens['prazo_item_valido'] = compra.notna() & prazo.notna() & prazo.ge(compra)
    itens['prazo_item_analise'] = prazo.where(itens['prazo_item_valido'])
    itens['item_despacho_avaliavel'] = (
        itens['prazo_item_valido'] & despacho.notna() & despacho.ge(compra)
        & itens['pedido_encontrado'] & ~itens['order_status'].isin(['canceled', 'unavailable'])
    )
    itens['item_atrasado'] = itens['item_despacho_avaliavel'] & despacho.gt(prazo)
    itens['categoria_analise'] = itens['product_category_name'].fillna('sem_categoria')

    # Cada linha passa a representar um vendedor dentro de um pedido.
    campos_pedido = [
        'order_status', 'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date', 'seller_city', 'seller_state',
        'qtd_vendedores_pedido', 'nota_media_pedido', 'menor_nota_pedido',
        'maior_nota_pedido', 'qtd_avaliacoes_pedido', 'qtd_avaliacoes_validas',
        'qtd_avaliacoes_baixas_pedido', 'pedido_encontrado', 'vendedor_encontrado',
        'avaliacao_encontrada',
    ]
    agregacoes = {campo: (campo, 'first') for campo in campos_pedido}
    agregacoes.update({
        'qtd_itens': ('order_item_id', 'size'),
        'qtd_produtos_distintos': ('product_id', 'nunique'),
        'valor_produtos': ('price', lambda s: s.sum(min_count=1)),
        'valor_frete': ('freight_value', lambda s: s.sum(min_count=1)),
        'prazo_despacho_vendedor': ('prazo_item_analise', 'min'),
        'ultimo_prazo_despacho_vendedor': ('prazo_item_analise', 'max'),
        'qtd_itens_prazo_valido': ('prazo_item_valido', 'sum'),
        'qtd_itens_despacho_avaliavel': ('item_despacho_avaliavel', 'sum'),
        'qtd_itens_atrasados': ('item_atrasado', 'sum'),
        'qtd_itens_sem_produto': ('produto_encontrado', lambda s: (~s).sum()),
        'qtd_categorias': ('categoria_analise', 'nunique'),
        'categorias': ('categoria_analise', lambda s: '|'.join(sorted(s.unique()))),
        'despacho_antes_compra': ('despacho_antes_compra', 'max'),
        'prazo_antes_compra': ('prazo_antes_compra', 'max'),
    })
    tabela = itens.groupby(['seller_id', 'order_id'], as_index=False).agg(**agregacoes)
    tabela['valor_total'] = tabela['valor_produtos'] + tabela['valor_frete']
    tabela['pedido_multivendedor'] = tabela['qtd_vendedores_pedido'].gt(1)
    tabela['mes_compra'] = tabela['order_purchase_timestamp'].dt.strftime('%Y-%m')

    tabela['despacho_avaliavel'] = tabela['qtd_itens_despacho_avaliavel'].eq(tabela['qtd_itens'])
    diferenca_despacho = (
        tabela['order_delivered_carrier_date'] - tabela['prazo_despacho_vendedor']
    ).dt.total_seconds() / 86400
    tabela['diferenca_despacho_dias'] = diferenca_despacho.where(tabela['despacho_avaliavel'])
    tabela['atraso_despacho_dias'] = tabela['diferenca_despacho_dias'].clip(lower=0)
    tabela['flag_atraso_despacho'] = tabela['qtd_itens_atrasados'].gt(0).astype('boolean')
    tabela.loc[~tabela['despacho_avaliavel'], 'flag_atraso_despacho'] = pd.NA

    entrega = tabela['order_delivered_customer_date']
    estimativa = tabela['order_estimated_delivery_date']
    tabela['entrega_avaliavel'] = (
        tabela['order_status'].eq('delivered') & entrega.notna() & estimativa.notna()
        & tabela['order_purchase_timestamp'].notna()
        & entrega.ge(tabela['order_purchase_timestamp'])
        & estimativa.ge(tabela['order_purchase_timestamp'].dt.normalize())
    )
    # A previsão de entrega é uma data: a hora do recebimento no mesmo dia não é atraso.
    tabela['diferenca_entrega_dias'] = (entrega.dt.normalize() - estimativa.dt.normalize()).dt.days.where(tabela['entrega_avaliavel'])
    tabela['atraso_entrega_dias'] = tabela['diferenca_entrega_dias'].clip(lower=0)
    tabela['flag_atraso_entrega'] = tabela['diferenca_entrega_dias'].gt(0).astype('boolean')
    tabela.loc[~tabela['entrega_avaliavel'], 'flag_atraso_entrega'] = pd.NA

    # Flags sem informação permanecem ausentes, em vez de virar avaliações favoráveis.
    tabela['flag_nota_baixa'] = tabela['nota_media_pedido'].le(2).astype('boolean')
    tabela.loc[tabela['nota_media_pedido'].isna(), 'flag_nota_baixa'] = pd.NA
    conferir_chave(tabela, ['seller_id', 'order_id'], 'tabela analítica')
    assert tabela['qtd_itens'].sum() == len(base_itens)
    assert len(tabela) == len(base_itens[['seller_id', 'order_id']].drop_duplicates())
    return tabela

In [45]:
# [IA] registro 1 do complemento das partes 3 e 4.
tabela_analitica = construir_tabela_analitica(base_itens_relacionada)

itens_entrada = dados_limpos['items'].drop_duplicates()
conferencia_totais = pd.DataFrame({
    'etapa': ['itens após retirar cópias completas', 'itens relacionados', 'vendedor-pedido'],
    'linhas': [len(itens_entrada), len(base_itens_relacionada), len(tabela_analitica)],
    'quantidade_itens': [len(itens_entrada), len(base_itens_relacionada), tabela_analitica['qtd_itens'].sum()],
    'valor_produtos': [itens_entrada['price'].sum(), base_itens_relacionada['price'].sum(), tabela_analitica['valor_produtos'].sum()],
    'valor_frete': [itens_entrada['freight_value'].sum(), base_itens_relacionada['freight_value'].sum(), tabela_analitica['valor_frete'].sum()],
})
display(conferencia_totais)

assert not tabela_analitica.duplicated(['seller_id', 'order_id']).any()
assert tabela_analitica['qtd_itens'].sum() == len(itens_entrada)
assert abs(tabela_analitica['valor_produtos'].sum() - itens_entrada['price'].sum()) < 0.000001
assert abs(tabela_analitica['valor_frete'].sum() - itens_entrada['freight_value'].sum()) < 0.000001

display(tabela_analitica[[
    'seller_id', 'order_id', 'seller_state', 'qtd_itens', 'valor_produtos',
    'prazo_despacho_vendedor', 'order_delivered_carrier_date',
    'flag_atraso_despacho', 'atraso_despacho_dias', 'nota_media_pedido',
    'pedido_multivendedor',
]].head(10))

,etapa,linhas,quantidade_itens,valor_produtos,valor_frete
0,itens após retirar cópias completas,112650,112650,13591643.7,2251909.54
1,itens relacionados,112650,112650,13591643.7,2251909.54
2,vendedor-pedido,100010,112650,13591643.7,2251909.54


,seller_id,order_id,seller_state,qtd_itens,valor_produtos,prazo_despacho_vendedor,order_delivered_carrier_date,flag_atraso_despacho,atraso_despacho_dias,nota_media_pedido,pedido_multivendedor
0,0015a82c2db000af6aaaf3ae2ecb0532,7f39ba4c9052be115350065d07583cac,SP,1,895.0,2017-10-24 23:56:20,2017-10-20 14:29:01,False,0.0,1.0,False
1,0015a82c2db000af6aaaf3ae2ecb0532,9dc8d1a6f16f1b89874c29c9d8d30447,SP,1,895.0,2017-10-18 14:49:22,2017-10-17 15:42:42,False,0.0,5.0,False
2,0015a82c2db000af6aaaf3ae2ecb0532,d455a8cb295653b55abda06d434ab492,SP,1,895.0,2017-10-12 22:24:16,2017-09-29 15:53:03,False,0.0,5.0,False
3,001cca7ae9ae17fb1caed9dfb1094831,006e43460a55bc60c0a437521e426529,ES,1,99.0,2017-05-17 01:30:22,2017-05-12 12:26:32,False,0.0,5.0,False
4,001cca7ae9ae17fb1caed9dfb1094831,00dfb074b5c910fbd08e04691c4b712f,ES,1,99.5,2017-06-19 19:55:19,2017-06-09 15:12:41,False,0.0,5.0,False
5,001cca7ae9ae17fb1caed9dfb1094831,0240c9d87266e4cabc3bacdcdc6e77e4,ES,1,129.0,2018-06-18 09:18:02,2018-06-12 14:18:00,False,0.0,5.0,False
6,001cca7ae9ae17fb1caed9dfb1094831,038345c4a9d21c17180f2f77e1701c69,ES,1,99.0,2017-11-20 20:07:41,2017-11-16 18:48:59,False,0.0,NaN,False
7,001cca7ae9ae17fb1caed9dfb1094831,03c765107c8fc34c68a82bbd6c686856,ES,1,112.0,2017-04-13 20:25:10,2017-04-10 12:26:52,False,0.0,5.0,False
8,001cca7ae9ae17fb1caed9dfb1094831,06c07d5b2696b9631375e2877c955b3b,ES,1,99.0,2017-11-13 18:15:29,2017-11-08 19:28:44,False,0.0,4.0,False
9,001cca7ae9ae17fb1caed9dfb1094831,072d46cbc49c2ee26ddde8f25d22f617,ES,1,129.0,2017-09-18 22:10:19,2017-09-14 16:24:01,False,0.0,5.0,False


### Leitura dos campos de prazo e avaliação

- `prazo_despacho_vendedor`: menor prazo válido entre os itens daquele vendedor no pedido. `qtd_itens_atrasados` permite conferir quantos itens ultrapassaram seus próprios prazos.
- `diferenca_despacho_dias`: diferença entre despacho e prazo, em dias fracionários. Um atraso de 12 horas é 0,5 dia, sem arredondamento para zero.
- `atraso_despacho_dias`: parte positiva dessa diferença; zero representa despacho no prazo ou antecipado.
- `despacho_avaliavel`: exige os prazos de todos os itens, data de compra, despacho observado e cronologia válida. Pedidos cancelados ou indisponíveis não entram nesse denominador.
- `flag_atraso_despacho`: ausente quando o despacho não pode ser avaliado. Ausência de data não significa pontualidade.
- `flag_atraso_entrega`: compara o dia do recebimento com o dia estimado. Receber à noite no próprio dia previsto não é atraso de entrega.
- `flag_nota_baixa`: indica que a **média das avaliações do pedido** foi menor ou igual a 2. Não significa que qualquer avaliação isolada tenha sido baixa.

O resumo por vendedor usa o número de pedidos avaliáveis como denominador da taxa de atraso e o número de pedidos com nota como denominador da taxa de notas baixas.
O campo `nota_media` é a média das médias por pedido, dando o mesmo peso a cada pedido do vendedor.

In [46]:
# [IA] registro 1 do complemento das partes 3 e 4.
def resumir_vendedores(tabela_analitica):
    resumo = tabela_analitica.groupby('seller_id', as_index=False).agg(
        seller_state=('seller_state', 'first'),
        qtd_pedidos=('order_id', 'nunique'),
        qtd_itens=('qtd_itens', 'sum'),
        pedidos_despacho_avaliavel=('despacho_avaliavel', 'sum'),
        pedidos_com_atraso_despacho=('flag_atraso_despacho', 'sum'),
        atraso_medio_despacho_dias=('atraso_despacho_dias', 'mean'),
        pedidos_com_nota=('nota_media_pedido', 'count'),
        nota_media=('nota_media_pedido', 'mean'),
        pedidos_nota_baixa=('flag_nota_baixa', 'sum'),
        pedidos_multivendedor=('pedido_multivendedor', 'sum'),
    )
    resumo['taxa_atraso_despacho_pct'] = 100 * resumo['pedidos_com_atraso_despacho'] / resumo['pedidos_despacho_avaliavel'].replace(0, float('nan'))
    resumo['taxa_nota_baixa_pct'] = 100 * resumo['pedidos_nota_baixa'] / resumo['pedidos_com_nota'].replace(0, float('nan'))
    return resumo

In [47]:
# [IA] registro 1 do complemento das partes 3 e 4.
resumo_vendedores = resumir_vendedores(tabela_analitica)
display(resumo_vendedores.head(10))

conferencia_cobertura = pd.DataFrame({
    'verificacao': [
        'pares vendedor-pedido', 'pedidos distintos', 'vendedores distintos',
        'pares com despacho avaliável', 'pares sem despacho avaliável',
        'pares sem avaliação', 'pedidos com mais de um vendedor',
    ],
    'quantidade': [
        len(tabela_analitica), tabela_analitica['order_id'].nunique(),
        tabela_analitica['seller_id'].nunique(), tabela_analitica['despacho_avaliavel'].sum(),
        (~tabela_analitica['despacho_avaliavel']).sum(), tabela_analitica['nota_media_pedido'].isna().sum(),
        tabela_analitica.loc[tabela_analitica['pedido_multivendedor'], 'order_id'].nunique(),
    ],
})
display(conferencia_cobertura)

,seller_id,seller_state,qtd_pedidos,qtd_itens,pedidos_despacho_avaliavel,pedidos_com_atraso_despacho,atraso_medio_despacho_dias,pedidos_com_nota,nota_media,pedidos_nota_baixa,pedidos_multivendedor,taxa_atraso_despacho_pct,taxa_nota_baixa_pct
0,0015a82c2db000af6aaaf3ae2ecb0532,SP,3,3,3,0,0.000000,3,3.666667,1,0,0.0,33.333333
1,001cca7ae9ae17fb1caed9dfb1094831,ES,200,239,200,12,0.076053,197,3.984772,32,1,6.0,16.243655
2,001e6ad469a905060d959994f1b41e4f,RJ,1,1,0,0,NaN,1,1.000000,1,0,<NA>,100.0
3,002100f778ceb8431b7a1020ff7ab48f,SP,51,55,51,4,0.334108,51,3.901961,8,2,7.843137,15.686275
4,003554e2dce176b5555353e4f3555ac8,GO,1,1,1,0,0.000000,1,5.000000,0,0,0.0,0.0
5,004c9cd9d87a3c30c522c48c4fc07416,SP,158,170,158,0,0.000000,155,4.148387,19,1,0.0,12.258065
6,00720abe85ba0859807595bbf045a33b,SP,13,26,13,3,0.502090,13,3.615385,3,1,23.076923,23.076923
7,00ab3eff1b5192e5f1a63bcecfee11c8,SP,1,1,1,0,0.000000,1,5.000000,0,0,0.0,0.0
8,00d8b143d12632bad99c0ad66ad52825,MG,1,1,1,0,0.000000,1,5.000000,0,0,0.0,0.0
9,00ee68308b45bc5e2660cd833c3f81cc,SP,135,172,135,1,0.011556,134,4.298507,15,5,0.740741,11.19403


,verificacao,quantidade
0,pares vendedor-pedido,100010
1,pedidos distintos,98666
2,vendedores distintos,3095
3,pares com despacho avaliável,98749
4,pares sem despacho avaliável,1261
5,pares sem avaliação,763
6,pedidos com mais de um vendedor,1278


### Cuidados ao usar as saídas na etapa de análise

A unidade da tabela principal é vendedor–pedido. Para contar pedidos da plataforma inteira, use `order_id.nunique()`.
As notas de um pedido com vários vendedores aparecem associadas a cada vendedor envolvido; somar esses registros não equivale a contar avaliações distintas da plataforma.
O CSV de itens continua disponível para análises que precisem de produto ou categoria, respeitando a diferença entre os níveis.

Estas etapas preparam os dados. As respostas às três perguntas do briefing, os critérios de acompanhamento e as conclusões ficam para a análise da equipe.

In [48]:
# [IA] registro 1 do complemento das partes 3 e 4.
descricao_colunas = {'seller_id': ('Identificador do vendedor.', 'items/sellers'),
 'order_id': ('Identificador do pedido.', 'items/orders'),
 'order_status': ('Situação registrada para o pedido.', 'orders'),
 'order_purchase_timestamp': ('Data e hora da compra.', 'orders'),
 'order_approved_at': ('Data e hora da aprovação.', 'orders'),
 'order_delivered_carrier_date': ('Data e hora de entrega à transportadora, no nível do pedido.', 'orders'),
 'order_delivered_customer_date': ('Data e hora de recebimento pelo cliente.', 'orders'),
 'order_estimated_delivery_date': ('Data estimada de entrega ao cliente.', 'orders'),
 'seller_city': ('Cidade do cadastro do vendedor.', 'sellers'),
 'seller_state': ('UF do cadastro do vendedor.', 'sellers'),
 'qtd_vendedores_pedido': ('Vendedores distintos no pedido, considerando os itens carregados.', 'items'),
 'nota_media_pedido': ('Média das notas válidas de 1 a 5 do pedido.', 'reviews'),
 'menor_nota_pedido': ('Menor nota válida do pedido.', 'reviews'),
 'maior_nota_pedido': ('Maior nota válida do pedido.', 'reviews'),
 'qtd_avaliacoes_pedido': ('Registros de avaliação do pedido, após retirar cópias completas.', 'reviews'),
 'qtd_avaliacoes_validas': ('Registros do pedido com nota em 1, 2, 3, 4 ou 5.', 'reviews'),
 'qtd_avaliacoes_baixas_pedido': ('Registros de avaliação do pedido com nota 1 ou 2.', 'reviews'),
 'pedido_encontrado': ('Indica correspondência do item no cadastro de pedidos.', 'merge items/orders'),
 'vendedor_encontrado': ('Indica correspondência no cadastro de vendedores.', 'merge items/sellers'),
 'avaliacao_encontrada': ('Indica que houve registro de avaliação para o pedido.', 'merge items/reviews'),
 'qtd_itens': ('Quantidade de linhas de itens no par vendedor-pedido.', 'items'),
 'qtd_produtos_distintos': ('Produtos distintos do vendedor naquele pedido.', 'items'),
 'valor_produtos': ('Soma dos preços dos itens do vendedor no pedido, em reais.', 'items.price'),
 'valor_frete': ('Soma dos fretes dos itens do vendedor no pedido, em reais.', 'items.freight_value'),
 'prazo_despacho_vendedor': ('Menor shipping_limit_date válido do vendedor no pedido.', 'items'),
 'ultimo_prazo_despacho_vendedor': ('Maior shipping_limit_date válido do vendedor no pedido.', 'items'),
 'qtd_itens_prazo_valido': ('Itens com prazo preenchido e não anterior à compra.', 'items/orders'),
 'qtd_itens_despacho_avaliavel': ('Itens com despacho e prazo comparáveis, fora de cancelados/indisponíveis.',
                                  'items/orders'),
 'qtd_itens_atrasados': ('Itens avaliáveis com despacho posterior ao seu próprio prazo.', 'items/orders'),
 'qtd_itens_sem_produto': ('Itens cujo product_id não encontrou cadastro.', 'merge items/products'),
 'qtd_categorias': ('Categorias distintas no par, incluindo sem_categoria quando necessário.', 'products'),
 'categorias': ('Categorias distintas ordenadas e separadas por |; ausências usam sem_categoria.',
                'products'),
 'despacho_antes_compra': ('Indica data de despacho anterior à compra.', 'orders'),
 'prazo_antes_compra': ('Indica algum prazo de item anterior à compra.', 'items/orders'),
 'valor_total': ('Valor dos produtos mais o frete do vendedor naquele pedido.', 'campos derivados'),
 'pedido_multivendedor': ('Indica mais de um vendedor no pedido entre os itens carregados.', 'items'),
 'mes_compra': ('Ano e mês da compra, no formato AAAA-MM.', 'orders'),
 'despacho_avaliavel': ('Todos os itens do par permitem avaliar o despacho.', 'campos derivados'),
 'diferenca_despacho_dias': ('Despacho menos menor prazo válido; dias fracionários, com sinal.',
                             'items/orders'),
 'atraso_despacho_dias': ('Diferença de despacho limitada a zero quando negativa; ausente se não avaliável.',
                          'campos derivados'),
 'flag_atraso_despacho': ('Verdadeiro se algum item atrasou; ausente se o par não é avaliável.',
                          'campos derivados'),
 'entrega_avaliavel': ('Pedido entregue, com compra, estimativa e recebimento em cronologia válida.',
                       'orders'),
 'diferenca_entrega_dias': ('Dia do recebimento menos dia estimado, com sinal.', 'orders'),
 'atraso_entrega_dias': ('Diferença de entrega limitada a zero quando negativa.', 'campos derivados'),
 'flag_atraso_entrega': ('Recebimento em dia posterior ao previsto; ausente se não avaliável.',
                         'campos derivados'),
 'flag_nota_baixa': ('Média das notas do pedido menor ou igual a 2; ausente se não há nota.', 'reviews')}

dicionario_tabela_analitica = pd.DataFrame([
    {'coluna': coluna, 'tipo': str(tabela_analitica[coluna].dtype),
     'descricao': descricao_colunas[coluna][0], 'origem': descricao_colunas[coluna][1]}
    for coluna in tabela_analitica.columns
])
display(dicionario_tabela_analitica)

,coluna,tipo,descricao,origem
0,seller_id,object,Identificador do vendedor.,items/sellers
1,order_id,object,Identificador do pedido.,items/orders
2,order_status,object,Situação registrada para o pedido.,orders
3,order_purchase_timestamp,datetime64[ns],Data e hora da compra.,orders
4,order_approved_at,datetime64[ns],Data e hora da aprovação.,orders
5,order_delivered_carrier_date,datetime64[ns],"Data e hora de entrega à transportadora, no ní...",orders
6,order_delivered_customer_date,datetime64[ns],Data e hora de recebimento pelo cliente.,orders
7,order_estimated_delivery_date,datetime64[ns],Data estimada de entrega ao cliente.,orders
8,seller_city,object,Cidade do cadastro do vendedor.,sellers
9,seller_state,object,UF do cadastro do vendedor.,sellers


In [49]:
# [IA] registro 1 do complemento das partes 3 e 4.
from pathlib import Path

pasta_saida_3_4 = Path(PASTA_TRATADOS) / 'partes_3_4'
pasta_saida_3_4.mkdir(parents=True, exist_ok=True)

saidas_3_4 = {
    'itens_relacionados.csv': base_itens_relacionada,
    'tabela_analitica_vendedor_pedido.csv': tabela_analitica,
    'resumo_vendedores.csv': resumo_vendedores,
    'dicionario_tabela_analitica.csv': dicionario_tabela_analitica,
    'auditoria_merges.csv': auditoria_relacionamentos['merges'],
    'auditoria_duplicatas.csv': auditoria_relacionamentos['duplicatas'],
    'conferencia_totais.csv': conferencia_totais,
    'conferencia_cobertura.csv': conferencia_cobertura,
}
for nome, df in saidas_3_4.items():
    destino = pasta_saida_3_4 / nome
    df.to_csv(destino, index=False, encoding='utf-8-sig')
    print(f'{nome}: {len(df):,} linhas — {destino.stat().st_size / 1024**2:.2f} MB')

print('Pasta de saída:', pasta_saida_3_4)
assert (pasta_saida_3_4 / 'tabela_analitica_vendedor_pedido.csv').stat().st_size < 100 * 1024**2

itens_relacionados.csv: 112,650 linhas — 42.56 MB


tabela_analitica_vendedor_pedido.csv: 100,010 linhas — 38.09 MB
resumo_vendedores.csv: 3,095 linhas — 0.27 MB
dicionario_tabela_analitica.csv: 46 linhas — 0.00 MB
auditoria_merges.csv: 4 linhas — 0.00 MB
auditoria_duplicatas.csv: 5 linhas — 0.00 MB
conferencia_totais.csv: 3 linhas — 0.00 MB
conferencia_cobertura.csv: 7 linhas — 0.00 MB
Pasta de saída: dados_tratados/partes_3_4


As saídas das partes 3 e 4 ficam em `dados_tratados/partes_3_4/`.
A função `relacionar_tabelas()` devolve os itens relacionados e a auditoria;
`construir_tabela_analitica()` devolve a base vendedor–pedido;
`resumir_vendedores()` produz um resumo auxiliar.
Essas funções podem ser chamadas na futura função `preprocessar()` da equipe.